In [17]:
import openai
import instructor

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient

### Mock example

In [1]:
prompt = """
You are a helpful assistant.
Return an answer to the question.
Question: what is you name?
"""

In [4]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

My name is ChatGPT.


### Instructor - Structured Outputs

In [6]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [8]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [9]:
response = client.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [10]:
response

Answer(answer='My name is ChatGPT.')

In [11]:
response, raw_response = client.create_with_completion(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [13]:
response

Answer(answer='My name is ChatGPT.')

In [12]:
raw_response

Response(id='resp_05b9da4cd0d58cc80069dca6b63b58819cae0545f19ca1cbd6', created_at=1776068278.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"My name is ChatGPT."}', call_id='call_AsqwiqbyQ0pEvsxUckLIAFdt', name='Answer', type='function_call', id='fc_05b9da4cd0d58cc80069dca6b71b30819c8b4f2510e47a89dd', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question.', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description=None)], top_p=0.98, background=False, completed_at=1776068279.0, conversation=None, max_output_tokens=None, max_

In [14]:
class AnswerWithReasoning(BaseModel):
    answer: str = Field(description="Answer to the question.")
    reasoning: str = Field(description="Reasoning for the answer.")

In [15]:
response = client.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [16]:
response

AnswerWithReasoning(answer='My name is ChatGPT.', reasoning="The user is asking for the assistant's name. As ChatGPT, I should identify myself clearly and concisely.")

### RAG Pipeline

In [19]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [20]:
def get_embedding(text, model='text-embedding-3-small'):

    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):
    
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=RAGGenerationResponse
)

    return response

def RAG_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocess_context = process_context(retrieved_context)
    prompt = build_prompt(preprocess_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [21]:
output = RAG_pipeline("Can I buy a fiction book?")

In [22]:
output

{'data_object': RAGGenerationResponse(answer='Yes. The available products include several fiction books, such as The Man His Means & His Methods, Devil of Dublin, Shorn, Monsters of Midlife, and Faultless Love.'),
 'answer': 'Yes. The available products include several fiction books, such as The Man His Means & His Methods, Devil of Dublin, Shorn, Monsters of Midlife, and Faultless Love.',
 'question': 'Can I buy a fiction book?',
 'retrieved_context_ids': ['B0BLFT3P9C',
  'B0BFW7MV1C',
  'B09XGZV7YT',
  'B0B6LSBPR1',
  '1737747227'],
 'retrieved_context': ['The Man His Means & His Methods The hardcover version of this beautiful novel includes over 75 original two-page spread, full-color works of art, designed by the author with AI . (Paperback is black and white with one page chapter art.) Conspiracy theories meet science FACTion in this novel about all the topics we have been discouraged from questioning! Questions like... What if inter-dimensional entities and extra-terrestrial bein